# Semana 7: Expresiones Regulares - Avanzado

## Programación para Ciencia de Datos
### Instituto Politécnico Nacional
### Febrero - Julio 2026

---

## Objetivos de Aprendizaje

Al finalizar esta semana, serás capaz de:

1. **Aplicar** lookahead y lookbehind para validaciones complejas
2. **Utilizar** banderas (flags) para modificar el comportamiento de regex
3. **Optimizar** expresiones regulares con `re.compile()`
4. **Construir** patrones avanzados para extracción de datos

## 1. Lookahead y Lookbehind (Aserciones)

Las **aserciones** permiten verificar condiciones sin consumir caracteres. Son como "mirar adelante" o "mirar atrás" sin moverse.

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    LOOKAHEAD Y LOOKBEHIND                               │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   LOOKAHEAD (mirar adelante)                                            │
│   ─────────────────────────                                             │
│   (?=...)   Positivo: coincide si lo siguiente ES el patrón             │
│   (?!...)   Negativo: coincide si lo siguiente NO ES el patrón          │
│                                                                         │
│   LOOKBEHIND (mirar atrás)                                              │
│   ────────────────────────                                              │
│   (?<=...)  Positivo: coincide si lo anterior ES el patrón              │
│   (?<!...)  Negativo: coincide si lo anterior NO ES el patrón           │
│                                                                         │
├─────────────────────────────────────────────────────────────────────────┤
│   IMPORTANTE: Las aserciones NO consumen caracteres                     │
│   Son "verificaciones" que no avanzan la posición del cursor            │
└─────────────────────────────────────────────────────────────────────────┘
```

### 1.1 Lookahead Positivo `(?=...)`

Coincide si **lo que sigue** es el patrón especificado.

```
┌─────────────────────────────────────────────────────────────────────┐
│   Ejemplo: \d+(?=€)                                                 │
│                                                                     │
│   Texto: "Cuesta 100€ o 200$"                                       │
│                                                                     │
│          C u e s t a   1 0 0 €   o   2 0 0 $                        │
│                        ─────┬─                                      │
│                             │                                       │
│                        "100" ✓ (seguido de €)                       │
│                        "200" ✗ (seguido de $, no €)                 │
│                                                                     │
│   Resultado: ['100']                                                │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
import re

# Lookahead positivo: números seguidos de €
texto = "Cuesta 100€ o 200$ o 150€"
patron = r'\d+(?=€)'

resultado = re.findall(patron, texto)
print(f"Números seguidos de €: {resultado}")

# Nota: el € NO está incluido en el resultado
# porque el lookahead no consume caracteres

In [ ]:
# Ejemplo práctico: encontrar palabras seguidas de signos de puntuación
texto = "Hola, mundo! ¿Cómo estás? Bien."

# Palabras seguidas de coma, punto, signo de interrogación o exclamación
patron = r'\w+(?=[,\.!?])'
palabras = re.findall(patron, texto)
print(f"Palabras antes de puntuación: {palabras}")

### 1.2 Lookahead Negativo `(?!...)`

Coincide si **lo que sigue NO es** el patrón especificado.

In [ ]:
# Lookahead negativo: números NO seguidos de €
texto = "Cuesta 100€ o 200$ o 150€ o 300"
patron = r'\d+(?!€)'

resultado = re.findall(patron, texto)
print(f"Números NO seguidos de €: {resultado}")

# Nota: puede coincidir parcialmente con números

In [ ]:
# Ejemplo más preciso: usar límite de palabra
texto = "Precios: 100€, 200USD, 150€, 300MXN"

# Números completos no seguidos de €
patron = r'\b\d+(?!€)\b'
resultado = re.findall(patron, texto)
print(f"Resultado con límites: {resultado}")

# Alternativa: números seguidos de moneda que NO sea €
patron2 = r'\d+(?=[A-Z]{3})'
resultado2 = re.findall(patron2, texto)
print(f"Números con código de moneda: {resultado2}")

### 1.3 Lookbehind Positivo `(?<=...)`

Coincide si **lo anterior es** el patrón especificado.

```
┌─────────────────────────────────────────────────────────────────────┐
│   Ejemplo: (?<=\$)\d+                                               │
│                                                                     │
│   Texto: "Precio: $500 o €300"                                      │
│                                                                     │
│          P r e c i o :   $ 5 0 0   o   € 3 0 0                      │
│                          ─┬─────                                    │
│                           │                                         │
│                      "500" ✓ (precedido por $)                      │
│                      "300" ✗ (precedido por €, no $)                │
│                                                                     │
│   Resultado: ['500']                                                │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Lookbehind positivo: números precedidos por $
texto = "Precio: $500 o €300 o $750"
patron = r'(?<=\$)\d+'

resultado = re.findall(patron, texto)
print(f"Números después de $: {resultado}")

In [ ]:
# Ejemplo práctico: extraer valores después de etiquetas
texto = "Nombre: Juan, Edad: 25, Ciudad: CDMX"

# Extraer valor después de "Edad: "
patron = r'(?<=Edad: )\d+'
edad = re.search(patron, texto)
print(f"Edad encontrada: {edad.group() if edad else 'No encontrada'}")

# Extraer valor después de "Ciudad: "
patron2 = r'(?<=Ciudad: )\w+'
ciudad = re.search(patron2, texto)
print(f"Ciudad encontrada: {ciudad.group() if ciudad else 'No encontrada'}")

### 1.4 Lookbehind Negativo `(?<!...)`

Coincide si **lo anterior NO es** el patrón especificado.

In [ ]:
# Lookbehind negativo: números NO precedidos por $
texto = "Cantidades: $500, 300, $750, 100"
patron = r'(?<!\$)\b\d+\b'

resultado = re.findall(patron, texto)
print(f"Números sin $ antes: {resultado}")

In [ ]:
# Ejemplo: encontrar palabras que NO empiecen después de punto
texto = "Primera oración. Segunda oración. Tercera"

# Palabras que NO siguen a un punto y espacio
patron = r'(?<!\. )\b[A-Z][a-z]+\b'
palabras = re.findall(patron, texto)
print(f"Palabras no después de punto: {palabras}")

### 1.5 Caso Práctico: Validación de Contraseña

Las aserciones son perfectas para validar múltiples condiciones:

In [ ]:
def validar_password(password):
    """
    Valida que la contraseña cumpla:
    - Al menos 8 caracteres
    - Al menos una mayúscula
    - Al menos una minúscula
    - Al menos un dígito
    - Al menos un carácter especial
    """
    patron = r'^(?=.*[A-Z])(?=.*[a-z])(?=.*\d)(?=.*[!@#$%^&*]).{8,}$'
    return bool(re.match(patron, password))

# Pruebas
passwords = [
    "Abc123!@",      # Válida
    "abc123!@",      # Sin mayúscula
    "ABC123!@",      # Sin minúscula
    "Abcdefg!",      # Sin dígito
    "Abc12345",      # Sin especial
    "Ab1!",          # Muy corta
]

print("Validación de contraseñas:")
print("-" * 40)
for pwd in passwords:
    estado = "✓ Válida" if validar_password(pwd) else "✗ Inválida"
    print(f"  {pwd:<15} {estado}")

## 2. Banderas (Flags)

Las banderas modifican el comportamiento de las expresiones regulares.

```
┌─────────────────────────────────────────────────────────────────────────┐
│                          BANDERAS (FLAGS)                               │
├─────────────┬───────────────────────────────────────────────────────────┤
│ Bandera     │ Descripción                                               │
├─────────────┼───────────────────────────────────────────────────────────┤
│ re.I        │ IGNORECASE - Ignora mayúsculas/minúsculas                 │
│ re.M        │ MULTILINE - ^ y $ coinciden en cada línea                 │
│ re.S        │ DOTALL - El punto (.) coincide con \n también             │
│ re.X        │ VERBOSE - Permite comentarios y espacios en el patrón     │
│ re.A        │ ASCII - \w, \d, \s solo coinciden con ASCII               │
│ re.U        │ UNICODE - Comportamiento Unicode (default en Python 3)   │
└─────────────┴───────────────────────────────────────────────────────────┘

Uso: re.findall(patrón, texto, re.IGNORECASE | re.MULTILINE)
```

### 2.1 `re.IGNORECASE` (re.I)

In [ ]:
texto = "Python es genial. PYTHON es poderoso. python es fácil."

# Sin IGNORECASE
resultado_sin = re.findall(r'python', texto)
print(f"Sin IGNORECASE: {resultado_sin}")

# Con IGNORECASE
resultado_con = re.findall(r'python', texto, re.IGNORECASE)
print(f"Con IGNORECASE: {resultado_con}")

# También se puede usar re.I como abreviación
resultado_i = re.findall(r'python', texto, re.I)
print(f"Con re.I: {resultado_i}")

### 2.2 `re.MULTILINE` (re.M)

In [ ]:
texto = """Línea 1: inicio
Línea 2: medio
Línea 3: final"""

# Sin MULTILINE: ^ solo coincide al inicio del texto completo
resultado_sin = re.findall(r'^Línea \d', texto)
print(f"Sin MULTILINE: {resultado_sin}")

# Con MULTILINE: ^ coincide al inicio de cada línea
resultado_con = re.findall(r'^Línea \d', texto, re.MULTILINE)
print(f"Con MULTILINE: {resultado_con}")

In [ ]:
# Ejemplo práctico: extraer líneas que terminan con números
logs = """Error en línea 42
Warning en proceso
Fatal en línea 100
Info: todo bien"""

# Líneas que terminan con número
patron = r'^.+\d+$'
lineas = re.findall(patron, logs, re.M)
print("Líneas que terminan con número:")
for linea in lineas:
    print(f"  {linea}")

### 2.3 `re.DOTALL` (re.S)

In [ ]:
texto = """<div>
  Contenido
  multilínea
</div>"""

# Sin DOTALL: el punto NO coincide con \n
resultado_sin = re.search(r'<div>.*</div>', texto)
print(f"Sin DOTALL: {resultado_sin}")

# Con DOTALL: el punto SÍ coincide con \n
resultado_con = re.search(r'<div>.*</div>', texto, re.DOTALL)
print(f"Con DOTALL: {resultado_con.group() if resultado_con else None}")

### 2.4 `re.VERBOSE` (re.X) - Patrones Legibles

In [ ]:
# Patrón de email SIN verbose (difícil de leer)
patron_compacto = r'^[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}$'

# Mismo patrón CON verbose (mucho más legible)
patron_verbose = r'''
    ^                   # Inicio de la cadena
    [\w.+-]+            # Usuario: letras, números, puntos, +, -
    @                   # Arroba literal
    [\w-]+              # Dominio: letras, números, guiones
    \.                  # Punto literal
    [a-zA-Z]{2,}        # Extensión: al menos 2 letras
    $                   # Fin de la cadena
'''

email = "usuario@dominio.com"

# Ambos producen el mismo resultado
print(f"Compacto: {bool(re.match(patron_compacto, email))}")
print(f"Verbose:  {bool(re.match(patron_verbose, email, re.VERBOSE))}")

In [ ]:
# Ejemplo: patrón para número de teléfono mexicano
patron_telefono = r'''
    ^                       # Inicio
    (?:\+52\s?)?            # Código de país opcional (+52)
    (?:\(\d{2,3}\)\s?)?     # Código de área entre paréntesis opcional
    \d{2,4}                 # Primera parte del número
    [-\s]?                  # Separador opcional
    \d{3,4}                 # Segunda parte
    [-\s]?                  # Separador opcional
    \d{4}                   # Última parte (4 dígitos)
    $                       # Fin
'''

telefonos = [
    "+52 55 1234 5678",
    "(55) 1234-5678",
    "55-1234-5678",
    "5512345678",
]

print("Validación de teléfonos:")
for tel in telefonos:
    valido = bool(re.match(patron_telefono, tel, re.VERBOSE))
    print(f"  {tel}: {'✓' if valido else '✗'}")

### 2.5 Combinando Banderas

In [ ]:
# Combinar múltiples banderas con |
texto = """PYTHON es genial
python es fácil
Python es popular"""

# IGNORECASE + MULTILINE
patron = r'^python'
resultado = re.findall(patron, texto, re.IGNORECASE | re.MULTILINE)
print(f"Con I + M: {resultado}")

## 3. Grupos Avanzados

### 3.1 Grupos No Capturadores `(?:...)`

Agrupan sin capturar (no aparecen en `groups()`).

In [ ]:
texto = "archivo.txt, imagen.jpg, documento.pdf"

# Con grupos capturadores normales
patron_captura = r'(\w+)\.(txt|jpg|pdf)'
resultado = re.findall(patron_captura, texto)
print(f"Con captura: {resultado}")

# Con grupo no capturador para las extensiones
patron_no_captura = r'(\w+)\.(?:txt|jpg|pdf)'
resultado = re.findall(patron_no_captura, texto)
print(f"Sin captura de extensión: {resultado}")

In [ ]:
# Útil para agrupar alternativas sin capturar
texto = "Tengo 3 gatos y 5 perros"

# Encontrar número + animal
patron = r'\d+\s+(?:gatos?|perros?)'
resultado = re.findall(patron, texto)
print(f"Resultado: {resultado}")

### 3.2 Backreferences (Referencias hacia atrás)

Permiten referenciar grupos capturados anteriormente.

In [ ]:
# Encontrar palabras duplicadas consecutivas
texto = "El el gato gato saltó muy muy alto"

# \1 referencia al primer grupo capturado
patron = r'\b(\w+)\s+\1\b'
duplicados = re.findall(patron, texto, re.IGNORECASE)
print(f"Palabras duplicadas: {duplicados}")

In [ ]:
# Encontrar tags HTML que coincidan (apertura y cierre)
html = "<div>contenido</div> <span>texto</span> <div>otro</p>"

# \1 asegura que el tag de cierre sea igual al de apertura
patron = r'<(\w+)>.*?</\1>'
tags_validos = re.findall(patron, html)
print(f"Tags con apertura/cierre correctos: {tags_validos}")

In [ ]:
# Backreference con nombre
texto = "usuario: admin, contraseña: admin"

# Encontrar donde usuario y contraseña son iguales (¡peligro!)
patron = r'usuario: (?P<user>\w+), contraseña: (?P=user)'
match = re.search(patron, texto)

if match:
    print(f"¡Alerta! Usuario y contraseña iguales: {match.group('user')}")

## 4. Greedy vs Non-Greedy (Codicioso vs Perezoso)

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    GREEDY VS NON-GREEDY                                 │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   GREEDY (codicioso) - Por defecto                                      │
│   ────────────────────────────────                                      │
│   *   +   ?   {n,m}   →  Coinciden lo MÁXIMO posible                    │
│                                                                         │
│   NON-GREEDY (perezoso) - Agregar ?                                     │
│   ────────────────────────────────────                                  │
│   *?  +?  ??  {n,m}?  →  Coinciden lo MÍNIMO posible                    │
│                                                                         │
├─────────────────────────────────────────────────────────────────────────┤
│   Ejemplo con texto: "<tag>contenido</tag>más<tag>otro</tag>"           │
│                                                                         │
│   Patrón greedy:     <tag>.*</tag>                                      │
│   Resultado:         "<tag>contenido</tag>más<tag>otro</tag>"           │
│                       └──────────────────────────────────────┘          │
│                                                                         │
│   Patrón non-greedy: <tag>.*?</tag>                                     │
│   Resultado:         "<tag>contenido</tag>" y "<tag>otro</tag>"         │
│                       └─────────────────┘     └────────────┘            │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
texto = "<div>primero</div><div>segundo</div>"

# Greedy: captura TODO desde el primer <div> hasta el último </div>
patron_greedy = r'<div>.*</div>'
resultado_greedy = re.findall(patron_greedy, texto)
print(f"Greedy:     {resultado_greedy}")

# Non-greedy: captura cada <div>...</div> por separado
patron_non_greedy = r'<div>.*?</div>'
resultado_non_greedy = re.findall(patron_non_greedy, texto)
print(f"Non-greedy: {resultado_non_greedy}")

In [ ]:
# Ejemplo práctico: extraer strings entre comillas
texto = 'Dijo "hola" y luego "adiós" antes de irse'

# Greedy - captura de la primera a la última comilla
greedy = re.findall(r'".*"', texto)
print(f"Greedy:     {greedy}")

# Non-greedy - captura cada string por separado
non_greedy = re.findall(r'".*?"', texto)
print(f"Non-greedy: {non_greedy}")

## 5. Compilación de Patrones con `re.compile()`

Cuando usas un patrón múltiples veces, compilarlo mejora el rendimiento.

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        re.compile()                                     │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   SIN compile (cada vez se compila el patrón):                          │
│   ─────────────────────────────────────────────                         │
│   for texto in textos:                                                  │
│       re.search(r'\d+', texto)  # Compila cada iteración               │
│                                                                         │
│   CON compile (se compila una sola vez):                                │
│   ──────────────────────────────────────                                │
│   patron = re.compile(r'\d+')   # Compila una vez                       │
│   for texto in textos:                                                  │
│       patron.search(texto)      # Reutiliza el patrón compilado        │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
import re
import time

# Crear un patrón compilado
patron_email = re.compile(
    r'^[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}$',
    re.IGNORECASE
)

# Usar el patrón compilado
emails = [
    "usuario@ejemplo.com",
    "otro@dominio.mx",
    "invalido@",
    "MAYUSCULAS@DOMINIO.COM"
]

print("Validación con patrón compilado:")
for email in emails:
    # Usar .match() directamente en el objeto compilado
    valido = bool(patron_email.match(email))
    print(f"  {email}: {'✓' if valido else '✗'}")

In [ ]:
# Comparación de rendimiento
texto = "El número 12345 aparece varias veces: 67890, 11111"
iteraciones = 100000

# Sin compile
inicio = time.time()
for _ in range(iteraciones):
    re.findall(r'\d+', texto)
tiempo_sin = time.time() - inicio

# Con compile
patron_compilado = re.compile(r'\d+')
inicio = time.time()
for _ in range(iteraciones):
    patron_compilado.findall(texto)
tiempo_con = time.time() - inicio

print(f"Sin compile: {tiempo_sin:.4f} segundos")
print(f"Con compile: {tiempo_con:.4f} segundos")
print(f"Mejora: {((tiempo_sin - tiempo_con) / tiempo_sin * 100):.1f}%")

## 6. Métodos Avanzados del Módulo `re`

### 6.1 `re.finditer()` - Iterador de coincidencias

In [ ]:
texto = "Precios: $100, $250, $75"
patron = r'\$(\d+)'

# finditer retorna un iterador de objetos Match
# Más eficiente en memoria que findall para textos grandes
print("Usando finditer:")
for match in re.finditer(patron, texto):
    print(f"  Encontrado: {match.group()} en posición {match.start()}-{match.end()}")
    print(f"  Valor capturado: {match.group(1)}")

### 6.2 `re.subn()` - Reemplazar y contar

In [ ]:
texto = "gato gato perro gato"

# subn retorna tupla (texto_modificado, número_de_reemplazos)
resultado, num_reemplazos = re.subn(r'gato', 'GATO', texto)

print(f"Original: {texto}")
print(f"Modificado: {resultado}")
print(f"Reemplazos realizados: {num_reemplazos}")

### 6.3 `re.sub()` con función de reemplazo

In [ ]:
# Reemplazar usando una función
texto = "Los precios son 100, 200 y 300 pesos"

def duplicar_precio(match):
    """Duplica el número encontrado."""
    numero = int(match.group())
    return str(numero * 2)

resultado = re.sub(r'\d+', duplicar_precio, texto)
print(f"Original:  {texto}")
print(f"Duplicado: {resultado}")

In [ ]:
# Ejemplo práctico: formatear nombres
nombres = "JUAN PÉREZ, MARÍA GARCÍA, PEDRO LÓPEZ"

def formatear_nombre(match):
    """Convierte 'NOMBRE APELLIDO' a 'Apellido, Nombre'."""
    partes = match.group().split()
    return f"{partes[1].title()}, {partes[0].title()}"

patron = r'[A-ZÁÉÍÓÚÑ]+\s+[A-ZÁÉÍÓÚÑ]+'
resultado = re.sub(patron, formatear_nombre, nombres)

print(f"Original:   {nombres}")
print(f"Formateado: {resultado}")

## 7. Patrones Complejos - Ejemplos Prácticos

### 7.1 Extractor de URLs

In [ ]:
patron_url = re.compile(r'''
    https?://                     # Protocolo http o https
    (?:www\.)?                    # www opcional
    [\w.-]+                       # Dominio
    (?:\.[a-zA-Z]{2,})+           # Extensión(es)
    (?:/[\w./-]*)?                # Path opcional
    (?:\?[\w=&]*)?                # Query string opcional
''', re.VERBOSE)

texto = """
Visita https://www.ejemplo.com/pagina?id=123
También http://otro-sitio.mx/ruta/archivo.html
Y https://api.servicio.com/v2/datos
"""

urls = patron_url.findall(texto)
print("URLs encontradas:")
for url in urls:
    print(f"  {url}")

### 7.2 Parser de Logs

In [ ]:
patron_log = re.compile(r'''
    ^                                   # Inicio de línea
    (?P<ip>\d{1,3}(?:\.\d{1,3}){3})    # Dirección IP
    \s+-\s+-\s+                         # Separadores
    \[(?P<fecha>[^\]]+)\]               # Fecha entre corchetes
    \s+"(?P<metodo>\w+)                 # Método HTTP
    \s+(?P<ruta>[^"]+)                  # Ruta solicitada
    \s+HTTP/[\d.]+"                     # Versión HTTP
    \s+(?P<status>\d{3})                # Código de estado
    \s+(?P<bytes>\d+|-)                 # Bytes transferidos
''', re.VERBOSE | re.MULTILINE)

logs = """192.168.1.1 - - [15/Mar/2024:10:23:45] "GET /index.html HTTP/1.1" 200 1234
10.0.0.5 - - [15/Mar/2024:10:24:00] "POST /api/datos HTTP/1.1" 201 567
192.168.1.100 - - [15/Mar/2024:10:25:30] "GET /imagen.jpg HTTP/1.1" 404 0"""

print("Análisis de logs:")
print("-" * 70)
for match in patron_log.finditer(logs):
    datos = match.groupdict()
    print(f"IP: {datos['ip']}")
    print(f"  Fecha: {datos['fecha']}")
    print(f"  {datos['metodo']} {datos['ruta']} -> {datos['status']}")
    print()

### 7.3 Validador de Tarjeta de Crédito

In [ ]:
def validar_tarjeta(numero):
    """
    Valida formato de número de tarjeta de crédito.
    Detecta: Visa, MasterCard, American Express
    """
    # Eliminar espacios y guiones
    numero_limpio = re.sub(r'[\s-]', '', numero)
    
    patrones = {
        'Visa': r'^4\d{15}$',                    # Empieza con 4, 16 dígitos
        'MasterCard': r'^5[1-5]\d{14}$',         # Empieza con 51-55, 16 dígitos
        'American Express': r'^3[47]\d{13}$',   # Empieza con 34/37, 15 dígitos
    }
    
    for tipo, patron in patrones.items():
        if re.match(patron, numero_limpio):
            return tipo, True
    
    return 'Desconocida', False

# Pruebas
tarjetas = [
    "4111 1111 1111 1111",    # Visa
    "5500-0000-0000-0004",    # MasterCard
    "378282246310005",        # American Express
    "1234567890123456",       # Inválida
]

print("Validación de tarjetas:")
for tarjeta in tarjetas:
    tipo, valida = validar_tarjeta(tarjeta)
    estado = f"✓ {tipo}" if valida else "✗ Inválida"
    print(f"  {tarjeta}: {estado}")

## 8. Buenas Prácticas y Errores Comunes

```
┌─────────────────────────────────────────────────────────────────────────┐
│                      BUENAS PRÁCTICAS                                   │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  ✓ Usa raw strings (r'...') siempre                                     │
│  ✓ Compila patrones que uses múltiples veces                            │
│  ✓ Usa re.VERBOSE para patrones complejos                               │
│  ✓ Prefiere non-greedy (*?, +?) cuando sea apropiado                    │
│  ✓ Usa grupos con nombre (?P<nombre>...) para claridad                  │
│  ✓ Prueba patrones en regex101.com antes de implementar                 │
│                                                                         │
├─────────────────────────────────────────────────────────────────────────┤
│                      ERRORES COMUNES                                    │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  ✗ Olvidar escapar metacaracteres (., *, +, ?, etc.)                    │
│  ✗ Confundir match() con search()                                       │
│  ✗ Usar greedy cuando necesitas non-greedy                              │
│  ✗ No anclar patrones (^ y $) cuando es necesario                       │
│  ✗ Patrones demasiado complejos (considera dividir)                     │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Ejemplo de error común: olvidar escapar el punto
texto = "archivo.txt y archivostxt"

# Incorrecto: el punto coincide con cualquier carácter
patron_mal = r'archivo.txt'
resultado_mal = re.findall(patron_mal, texto)
print(f"Sin escapar: {resultado_mal}")

# Correcto: el punto está escapado
patron_bien = r'archivo\.txt'
resultado_bien = re.findall(patron_bien, texto)
print(f"Con escape:  {resultado_bien}")

## 9. Ejercicios de Práctica

### Ejercicio 1: Extractor de menciones y hashtags

In [ ]:
# Extrae @menciones y #hashtags de un tweet
tweet = "Hola @usuario1 y @usuario2! Miren esto #Python #DataScience #IPN"

menciones = re.findall(r'@\w+', tweet)
hashtags = re.findall(r'#\w+', tweet)

print(f"Menciones: {menciones}")
print(f"Hashtags: {hashtags}")

### Ejercicio 2: Validador de RFC mexicano

In [ ]:
def validar_rfc(rfc):
    """
    Valida RFC mexicano.
    Persona física: 4 letras + 6 dígitos (fecha) + 3 homoclave
    Persona moral: 3 letras + 6 dígitos (fecha) + 3 homoclave
    """
    patron = r'''
        ^                           # Inicio
        [A-ZÑ&]{3,4}                # 3 o 4 letras (moral o física)
        \d{2}                       # Año (2 dígitos)
        (?:0[1-9]|1[0-2])           # Mes (01-12)
        (?:0[1-9]|[12]\d|3[01])     # Día (01-31)
        [A-Z\d]{3}                  # Homoclave
        $                           # Fin
    '''
    return bool(re.match(patron, rfc.upper(), re.VERBOSE))

# Pruebas
rfcs = [
    "GARC850101ABC",    # Persona física válido
    "XYZ850101AB1",     # Persona moral válido
    "ABC123456",        # Inválido
    "GARC851301ABC",    # Inválido (mes 13)
]

print("Validación de RFCs:")
for rfc in rfcs:
    estado = "✓ Válido" if validar_rfc(rfc) else "✗ Inválido"
    print(f"  {rfc}: {estado}")

### Ejercicio 3: Censurar información sensible

In [ ]:
def censurar_datos(texto):
    """Censura emails, teléfonos y números de tarjeta."""
    # Censurar emails
    texto = re.sub(
        r'[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}',
        '[EMAIL CENSURADO]',
        texto
    )
    
    # Censurar teléfonos (formato: XX-XXXX-XXXX o similar)
    texto = re.sub(
        r'\d{2,3}[-\s]?\d{4}[-\s]?\d{4}',
        '[TELÉFONO CENSURADO]',
        texto
    )
    
    # Censurar tarjetas (mostrar solo últimos 4 dígitos)
    def censurar_tarjeta(match):
        numero = re.sub(r'[\s-]', '', match.group())
        return f"****-****-****-{numero[-4:]}"
    
    texto = re.sub(
        r'\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}',
        censurar_tarjeta,
        texto
    )
    
    return texto

# Prueba
texto = """
Cliente: Juan Pérez
Email: juan.perez@email.com
Teléfono: 55-1234-5678
Tarjeta: 4111 1111 1111 1234
"""

print("Texto original:")
print(texto)
print("Texto censurado:")
print(censurar_datos(texto))

---

## Recursos Adicionales

- [Documentación oficial de `re`](https://docs.python.org/3/library/re.html)
- [Regex101](https://regex101.com/) - Herramienta online con explicaciones
- [Debuggex](https://www.debuggex.com/) - Visualizador de regex

---

## Próxima Semana

En la **Semana 8** comenzaremos con **Pandas: Introducción y Series**:
- Estructuras de datos: Series y DataFrames
- Creación y manipulación de Series
- Indexación y selección
- Operaciones básicas